# Nearest neighbourhood classifier

In one sentence, kNN looks for other objects in the neighborhood and assigns the most popular label to new objects. The $k$ in the name is the number of objects that we look for labels in the neighborhood.

In [ ]:
import numpy as np
import tensorflow_datasets as tfds
import pandas as pd
from math import sqrt
from sklearn.metrics import accuracy_score

# Construct a tf.data.Dataset
ds = tfds.load('titanic', split='train', shuffle_files=True)
titanic_df = tfds.as_dataframe(ds)

#titanic_df = pd.read_csv("titanic-train.csv")
titanic_df.drop("name", axis=1, inplace=True)
titanic_df.head()

columns = ['survived','age','fare']
features = columns[1:]
titanic_df = titanic_df[columns].replace([np.inf, -np.inf], np.nan).dropna()

titanic_df = titanic_df[titanic_df['fare'] > 30]

survivded_df = titanic_df[titanic_df['survived']==1].sample(50, random_state=12345)
not_survivded_df = titanic_df[titanic_df['survived']==0].sample(50, random_state=12345)

train_set = survivded_df.sample(40, random_state=12345)
train_set = pd.concat([train_set, not_survivded_df.sample(40, random_state=12345)])
plot_train_set = train_set
train_set_labels = train_set['survived']
train_set = train_set[features]


test_set = survivded_df.sample(10, random_state=12345)
test_set = pd.concat([test_set, not_survivded_df.sample(10, random_state=12345)])
plot_test_set = test_set
test_set_labels = test_set['survived']
test_set = test_set[features]

k = 5

def calculate_distance(x, v):
    """
    This method calculates the Euclidean distance between object x and v.

    :param x: first object
    :param v: second object
    :return: Euclidean distance
    """
    return sqrt((x[0] - v[0]) ** 2 + (x[1] - v[1]) ** 2)

def calculate_distance_matrix():
    """
    This method calculates the distance matrix between all objects.

    :return: A matrix of distances
    """    
    distance_matrix = np.zeros((len(data_set),len(data_set)))
    for i in range(len(data_set)):
        for j in range(len(data_set)):
            distance_matrix[i, j] = calculate_distance(data_set[i], data_set[j])
    return distance_matrix

def find_closest_objects(x, k):
    """
    Finds k closts objects to x.

    :return: A list of objects' ids.
    """        
    distances = []
    i = 0
    for item in train_set.values:
        distances.append([i, calculate_distance(x, item)])
        i = i + 1
    distances=np.array(distances)
    label_ids = distances[distances[:, 1].argsort()][:k,0]
    return [int(item) for item in label_ids]


def predict():
    """
    Assign a label of the most common label in the list of closest objects.

    :return: A list of predictions.
    """        
    predictions = []
    for item in test_set.values:
        label_ids = find_closest_objects(item, k)
        counts = np.bincount(train_set_labels.values[label_ids])
        label = np.argmax(counts)
        predictions.append(label)
    return predictions



predictions = predict()
acc = accuracy_score(predictions, test_set_labels)
print(acc)

In [ ]:
plot_train_set.loc[plot_train_set["survived"] == 1, "survived"] = "green"
plot_train_set.loc[plot_train_set["survived"] == 0, "survived"] = "red"

In [ ]:
plot_test_set['predicted'] = predictions

In [ ]:
plot_test_set.loc[(plot_test_set["survived"] == 1) & (plot_test_set["predicted"] == 0), "survived"] = "black"
plot_test_set.loc[(plot_test_set["survived"] == 1) & (plot_test_set["predicted"] == 1), "survived"] = "blue"
plot_test_set.loc[(plot_test_set["survived"] == 0) & (plot_test_set["predicted"] == 0), "survived"] = "orange"
plot_test_set.loc[(plot_test_set["survived"] == 0) & (plot_test_set["predicted"] == 1), "survived"] = "purple"
plot_test_set.drop(['predicted'],axis=1)

In [ ]:
plot_set = pd.concat([plot_test_set, plot_train_set])

In [ ]:
res = plot_set.plot.scatter(x='age',y='fare',c='survived').get_figure()

In [ ]:
res = plot_test_set.plot.scatter(x='age',y='fare',c='survived').get_figure()